# Practical worksheet: Diffusion and Latent Diffusion (MNIST)

## INSTRUCTOR VERSION (solutions + teaching notes)

## Learning focus
1. Train a basic noise-prediction diffusion model on MNIST.
2. Understand forward noising and reverse denoising steps.
3. Build a two-stage latent diffusion pipeline (autoencoder + diffusion in latent space).
4. Compare pixel-space diffusion vs latent-space diffusion behavior.

## Notebook setup
The next cells install/import dependencies, configure reproducibility, and define shared helpers.

In [ ]:
%pip install -q tqdm
%matplotlib inline

import random
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

Define reproducibility and device selection (`cuda`, `mps`, `cpu`).

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def get_device():
    if torch.cuda.is_available():
        return torch.device('cuda')
    if torch.backends.mps.is_available() and torch.backends.mps.is_built():
        return torch.device('mps')
    return torch.device('cpu')


set_seed(42)
device = get_device()
print('Device:', device)

Create MNIST loaders normalized to `[-1, 1]` and resized to 32x32.

In [ ]:
def build_mnist_loaders(batch_size=128, train_limit=None, test_limit=None, data_root='data', num_workers=0):
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])

    train_ds = datasets.MNIST(data_root, train=True, download=True, transform=transform)
    test_ds = datasets.MNIST(data_root, train=False, download=True, transform=transform)

    if train_limit is not None:
        train_ds = Subset(train_ds, list(range(min(train_limit, len(train_ds)))))
    if test_limit is not None:
        test_ds = Subset(test_ds, list(range(min(test_limit, len(test_ds)))))

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers)
    return train_loader, test_loader


Visualization helpers.

In [ ]:
def denorm(x):
    return (x + 1.0) / 2.0


def show_image_grid(images, channels=1, title='Images', n_show=25):
    images = images[:n_show].detach().cpu()
    images = denorm(images).clamp(0, 1)

    n = images.size(0)
    grid = int(np.ceil(np.sqrt(n)))
    fig, axes = plt.subplots(grid, grid, figsize=(grid * 1.6, grid * 1.6))
    axes = np.atleast_2d(axes)

    idx = 0
    for i in range(grid):
        for j in range(grid):
            ax = axes[i, j]
            ax.axis('off')
            if idx < n:
                if channels == 1:
                    ax.imshow(images[idx, 0], cmap='gray', vmin=0, vmax=1)
                else:
                    ax.imshow(images[idx].permute(1, 2, 0))
            idx += 1

    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


def plot_curve(values, title='Loss curve', ylabel='Loss'):
    plt.figure(figsize=(7, 4))
    plt.plot(values)
    plt.title(title)
    plt.xlabel('Epoch')
    plt.ylabel(ylabel)
    plt.grid(alpha=0.3)
    plt.show()

### PyTorch docs for key blocks used here
- `nn.Conv2d`: https://pytorch.org/docs/stable/generated/torch.nn.Conv2d.html
- `nn.ConvTranspose2d`: https://pytorch.org/docs/stable/generated/torch.nn.ConvTranspose2d.html
- `nn.Embedding`: https://pytorch.org/docs/stable/generated/torch.nn.Embedding.html

### Schedule and Forward Diffusion Helpers
This block defines the diffusion schedule and forward noising process.

Formulation:
- Choose `T` timesteps and linear noise schedule `beta_t`.
- `alpha_t = 1 - beta_t`
- $\bar{\alpha}_t = \prod_{s=1}^{t} \alpha_s$
- Forward process:
  $q(x_t \mid x_0) = \mathcal{N}\left(\sqrt{\bar{\alpha}_t}\,x_0,\,(1-\bar{\alpha}_t)I\right)$

Pseudocode:
1. Build `{beta_t, alpha_t, alpha_bar_t}`.
2. Sample noise `eps ~ N(0, I)`.
3. Construct noisy sample
   $x_t = \sqrt{\bar{\alpha}_t}\,x_0 + \sqrt{1-\bar{\alpha}_t}\,\varepsilon$.

### What `GaussianDiffusion` Does
`GaussianDiffusion` is the notebook's utility class for the full DDPM process. It does not learn parameters by itself; instead, it stores the schedule coefficients and applies the diffusion equations during training and sampling.

Main responsibilities:
- `__init__`: precompute `beta_t`, `alpha_t`, and cumulative products used throughout the algorithm.
- `q_sample(x_0, t)`: implement the forward process and create a noisy sample `x_t` from a clean input `x_0`.
- `p_sample(model, x_t, t)`: implement one reverse denoising step using the neural network prediction.
- `p_sample_loop(...)`: repeat reverse steps from pure noise until a final sample is produced.

Conceptually:
1. During training, we call `q_sample` to corrupt clean data and ask the model to predict the added noise.
2. During generation, we start from random Gaussian noise and repeatedly call `p_sample` to move toward a structured image or latent.

This means `GaussianDiffusion` handles the probabilistic mechanics, while the U-Net or latent denoiser learns the noise predictor.

In [ ]:
import torch

class GaussianDiffusion:
    """
    DDPM (Denoising Diffusion Probabilistic Models) Scheduler.
    """
    def __init__(self, num_timesteps=1000, beta_start=0.0001, beta_end=0.02, device='cpu'):
        self.num_timesteps = num_timesteps
        self.device = device
        
        # Linear beta scheduler (can be swapped for Cosine for better efficiency)
        self.betas = torch.linspace(beta_start, beta_end, num_timesteps).to(device)
        self.alphas = 1. - self.betas
        self.alphas_cumprod = torch.cumprod(self.alphas, dim=0)
        
        # alphas_cumprod_prev starts with 1.0 (no noise)
        self.alphas_cumprod_prev = torch.cat([torch.tensor([1.]).to(device), self.alphas_cumprod[:-1]])
        
        # Calculations for diffusion q(x_t | x_0)
        self.sqrt_alphas_cumprod = torch.sqrt(self.alphas_cumprod)
        self.sqrt_one_minus_alphas_cumprod = torch.sqrt(1. - self.alphas_cumprod)
        
        # Calculations for posterior q(x_{t-1} | x_t, x_0)
        # posterior_mean = posterior_mean_coef1 * x_0 + posterior_mean_coef2 * x_t
        self.posterior_mean_coef1 = self.betas * torch.sqrt(self.alphas_cumprod_prev) / (1. - self.alphas_cumprod)
        self.posterior_mean_coef2 = (1. - self.alphas_cumprod_prev) * torch.sqrt(self.alphas) / (1. - self.alphas_cumprod)
        
        # posterior_variance
        self.posterior_variance = self.betas * (1. - self.alphas_cumprod_prev) / (1. - self.alphas_cumprod)

    def q_sample(self, x_0, t, noise=None):
        """
        Forward diffusion process: Add noise to x_0 at step t.
        q(x_t | x_0) = N(x_t; sqrt(alpha_prod)*x_0, (1-alpha_prod)*I)
        """
        if noise is None:
            noise = torch.randn_like(x_0)
        
        sqrt_alpha_prod = self._get_index(self.sqrt_alphas_cumprod, t, x_0.shape)
        sqrt_one_minus_alpha_prod = self._get_index(self.sqrt_one_minus_alphas_cumprod, t, x_0.shape)
        
        return sqrt_alpha_prod * x_0 + sqrt_one_minus_alpha_prod * noise

    @torch.no_grad()
    def p_sample(self, model, x, t, t_index):
        """
        Reverse diffusion step: Sample x_{t-1} given x_t and the model.
        """
        betas_t = self._get_index(self.betas, t, x.shape)
        sqrt_one_minus_alpha_cumprod_t = self._get_index(self.sqrt_one_minus_alphas_cumprod, t, x.shape)
        sqrt_recip_alphas_t = 1. / torch.sqrt(self._get_index(self.alphas, t, x.shape))
        
        # Predict noise
        predicted_noise = model(x, t)
        
        # Compute mean
        model_mean = sqrt_recip_alphas_t * (x - betas_t * predicted_noise / sqrt_one_minus_alpha_cumprod_t)
        
        if t_index == 0:
            return model_mean
        else:
            posterior_variance_t = self._get_index(self.posterior_variance, t, x.shape)
            noise = torch.randn_like(x)
            # Clip step to be safe, or just add variance
            return model_mean + torch.sqrt(posterior_variance_t) * noise

    @torch.no_grad()
    def p_sample_loop(self, model, shape):
        """
        Sample all steps from pure noise to reconstruct an image in latent space.
        """
        model.eval()
        x = torch.randn(shape).to(self.device)
        # Reverse loop from T-1 back to 0
        for i in reversed(range(0, self.num_timesteps)):
            t = torch.full((shape[0],), i, dtype=torch.long).to(self.device)
            x = self.p_sample(model, x, t, i)
        return x

    def _get_index(self, tensor, t, x_shape):
        """Get value at index t and expand to match x_shape."""
        out = tensor.gather(-1, t)
        return out.view(t.shape[0], *((1,) * (len(x_shape) - 1)))




### Denoiser Network
This model predicts noise `eps_theta(x_t, t)` from noisy inputs.

Purpose:
- Use a compact U-Net so the model keeps both coarse digit structure and fine stroke detail.
- Inject timestep information at multiple resolutions.
- Recover cleaner MNIST digits in fewer epochs than a flat CNN.

About `SiLU`:
- `SiLU(x) = x * sigmoid(x)`.
- It is a smooth activation, often easier to optimize than ReLU in diffusion-style models.
- Negative inputs are not hard-clipped to zero, which helps preserve gradient flow.

Pseudocode:
1. Embed timestep `t`.
2. Encode noisy input through downsampling blocks.
3. Process a bottleneck representation at low resolution.
4. Decode through upsampling blocks while concatenating skip connections.
5. Predict a noise tensor with the same shape as the input.

In [ ]:
import torch
import torch.nn as nn
import math

class SinusoidalPosEmb(nn.Module):
    """
    Sinusoidal Position Embedding for time steps.
    """
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, x):
        device = x.device
        half_dim = self.dim // 2
        emb = math.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=device) * -emb)
        emb = x[:, None] * emb[None, :]
        emb = torch.cat((emb.sin(), emb.cos()), dim=-1)
        # Handle odd dimension by padding if necessary, but dim should be even
        return emb

class ResnetBlock(nn.Module):
    """
    Residual Block with Time Embedding projection.
    Supports channel dimension changes with short-cut projection.
    """
    def __init__(self, dim, time_emb_dim, out_dim=None):
        super().__init__()
        self.out_dim = out_dim or dim
        self.mlp = nn.Sequential(
            nn.SiLU(),
            nn.Linear(time_emb_dim, self.out_dim)
        )
        self.conv1 = nn.Conv2d(dim, self.out_dim, 3, padding=1)
        self.conv2 = nn.Conv2d(self.out_dim, self.out_dim, 3, padding=1)
        # GroupNorm tends to work better for diffusion than BatchNorm
        self.norm1 = nn.GroupNorm(4, dim)
        self.norm2 = nn.GroupNorm(4, self.out_dim)
        self.act = nn.SiLU()
        
        # Shortcut for residual if dims don't match
        self.shortcut = nn.Conv2d(dim, self.out_dim, 1) if dim != self.out_dim else nn.Identity()

    def forward(self, x, time_emb):
        h = self.norm1(x)
        h = self.act(h)
        h = self.conv1(h)
        # Add time embedding
        time_emb = self.mlp(time_emb)
        # Expand time_emb to match spatial dimensions [B, C, 1, 1]
        h = h + time_emb[:, :, None, None]
        h = self.norm2(h)
        h = self.act(h)
        h = self.conv2(h)
        return self.shortcut(x) + h


class LatentDenoiseNetwork(nn.Module):
    """
    Denoising Network operating on latent tensors of shape [B, latent_channels, H_lat, W_lat].
    For MNIST latents from VAE, shape may be [B, 4, 4, 4].
    """
    def __init__(self, latent_channels=4, model_channels=64, num_res_blocks=3):
        super().__init__()
        self.time_embed = nn.Sequential(
            SinusoidalPosEmb(model_channels),
            nn.Linear(model_channels, model_channels * 4),
            nn.SiLU(),
            nn.Linear(model_channels * 4, model_channels * 4),
        )
        
        self.init_conv = nn.Conv2d(latent_channels, model_channels, 3, padding=1)
        
        self.res_blocks = nn.ModuleList([
            ResnetBlock(model_channels, model_channels * 4)
            for _ in range(num_res_blocks)
        ])
        
        self.out_conv = nn.Conv2d(model_channels, latent_channels, 3, padding=1)
        
    def forward(self, x, t):
        # t is shape [B]
        t_emb = self.time_embed(t)
        h = self.init_conv(x)
        for block in self.res_blocks:
            h = block(h, t_emb)
        return self.out_conv(h)


# --- PIXEL UNET ---

class PixelUNet(nn.Module):
    """
    Standard UNet for Diffusion on image space.
    Fits 28x28 MNIST images.
    """
    def __init__(self, in_channels=1, model_channels=64):
        super().__init__()
        # Time Embedding
        self.time_embed = nn.Sequential(
            SinusoidalPosEmb(model_channels),
            nn.Linear(model_channels, model_channels * 4),
            nn.SiLU(),
            nn.Linear(model_channels * 4, model_channels * 4),
        )
        
        time_dim = model_channels * 4
        
        # Initial Conv
        self.init_conv = nn.Conv2d(in_channels, model_channels, 3, padding=1)
        
        # Down 1: 28 -> 14
        self.down1_res = ResnetBlock(model_channels, time_dim)
        self.down1_pool = nn.Conv2d(model_channels, model_channels, 3, stride=2, padding=1)
        
        # Down 2: 14 -> 7
        self.down2_res = ResnetBlock(model_channels, time_dim, out_dim=model_channels * 2)
        self.down2_pool = nn.Conv2d(model_channels * 2, model_channels * 2, 3, stride=2, padding=1)
        
        # Middle
        self.mid_res1 = ResnetBlock(model_channels * 2, time_dim)
        self.mid_res2 = ResnetBlock(model_channels * 2, time_dim)
        
        # Up 2: 7 -> 14
        self.up2_conv = nn.ConvTranspose2d(model_channels * 2, model_channels, 4, stride=2, padding=1) # 7 -> 14
        # Skip connection from down2_res is model_channels * 2
        # After concat: model_channels (up) + model_channels*2 (skip) = model_channels * 3
        self.up2_res = ResnetBlock(model_channels * 3, time_dim, out_dim=model_channels)
        
        # Up 1: 14 -> 28
        self.up1_conv = nn.ConvTranspose2d(model_channels, model_channels, 4, stride=2, padding=1) # 14 -> 28
        # Skip connection from down1_res is model_channels
        # After concat: model_channels (up) + model_channels (skip) = model_channels * 2
        self.up1_res = ResnetBlock(model_channels * 2, time_dim, out_dim=model_channels)
        
        # Out
        self.out_conv = nn.Conv2d(model_channels, in_channels, 3, padding=1)
        
    def forward(self, x, t):
        t_emb = self.time_embed(t)
        
        # Initial
        h_init = self.init_conv(x) # [B, C, 28, 28]
        
        # Down 1
        h1 = self.down1_res(h_init, t_emb) # [B, C, 28, 28]
        h1_pool = self.down1_pool(h1)      # [B, C, 14, 14]
        
        # Down 2
        h2 = self.down2_res(h1_pool, t_emb) # [B, 2C, 14, 14]
        h2_pool = self.down2_pool(h2)       # [B, 2C, 7, 7]
        
        # Middle
        h_mid = self.mid_res1(h2_pool, t_emb) # [B, 2C, 7, 7]
        h_mid = self.mid_res2(h_mid, t_emb)   # [B, 2C, 7, 7]
        
        # Up 2
        h_up2 = self.up2_conv(h_mid) # [B, C, 14, 14]
        h_up2 = torch.cat([h_up2, h2], dim=1) # [B, 3C, 14, 14]
        h_up2 = self.up2_res(h_up2, t_emb)   # [B, C, 14, 14]
        
        # Up 1
        h_up1 = self.up1_conv(h_up2) # [B, C, 28, 28]
        h_up1 = torch.cat([h_up1, h1], dim=1) # [B, 2C, 28, 28]
        h_up1 = self.up1_res(h_up1, t_emb)   # [B, C, 28, 28]
        
        # Out
        return self.out_conv(h_up1)




### Autoencoder for Latent Diffusion
This block defines the encoder/decoder used for the latent stage.

Purpose:
- Compress image space into smaller latent tensors without destroying digit structure.
- Run diffusion in latent space while still decoding recognizable strokes.

About `SiLU`:
- The autoencoder also uses `SiLU` because it gives smoother nonlinear transitions than ReLU.
- In practice this often helps reconstructions look less blocky or harsh.

Formulation:
- `z = E(x)`
- `x_hat = D(z)`
- Reconstruction objective: `L_rec = ||x_hat - x||^2`

In [ ]:
import torch
import torch.nn as nn

class Encoder(nn.Module):
    def __init__(self, latent_channels=4):
        super().__init__()
        self.net = nn.Sequential(
            # 1x28x28 -> 32x14x14
            nn.Conv2d(1, 32, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            # 32x14x14 -> 64x7x7
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            # 64x7x7 -> 64x4x4
            nn.Conv2d(64, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
        )
        # Latent projections
        # 64x4x4 -> latent_channels x 4x4 for mu and logvar
        self.mu = nn.Conv2d(64, latent_channels, kernel_size=1)
        self.logvar = nn.Conv2d(64, latent_channels, kernel_size=1)
        
    def forward(self, x):
        h = self.net(x)
        return self.mu(h), self.logvar(h)

class Decoder(nn.Module):
    def __init__(self, latent_channels=4):
        super().__init__()
        self.initial_conv = nn.Conv2d(latent_channels, 64, kernel_size=1)
        
        self.net = nn.Sequential(
            # 64x4x4 -> 64x7x7
            nn.ConvTranspose2d(64, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            # 64x7x7 -> 32x14x14
            nn.ConvTranspose2d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            # 32x14x14 -> 1x28x28
            nn.ConvTranspose2d(32, 1, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.Tanh() # Output range [-1, 1]
        )
        
    def forward(self, z):
        h = self.initial_conv(z)
        return self.net(h)

class VAE(nn.Module):
    def __init__(self, latent_channels=4):
        super().__init__()
        self.encoder = Encoder(latent_channels)
        self.decoder = Decoder(latent_channels)
        
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std
        
    def forward(self, x):
        mu, logvar = self.encoder(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decoder(z)
        return recon, mu, logvar




### Reverse Sampling Helper
This function runs reverse diffusion to generate samples from noise.

Approximate DDPM reverse update:
$$x_{t-1} = \frac{1}{\sqrt{\alpha_t}}\left(x_t - \frac{1-\alpha_t}{\sqrt{1-\bar{\alpha}_t}}\,\varepsilon_\theta(x_t,t)\right) + \sqrt{\beta_t}\,z$$
with $$z \sim \mathcal{N}(0, I)$$ for $$t>0$$, else $$z=0$$.

Pseudocode:
1. Start from Gaussian noise `x_T`.
2. For `t = T-1 ... 0`:
   - predict noise with model
   - apply reverse update
3. Return `x_0` samples.

In [ ]:
@torch.no_grad()
def sample_diffusion(model, schedule, shape):
    model.eval()
    x = torch.randn(shape, device=device)

    for step in reversed(range(schedule['T'])):
        t = torch.full((shape[0],), step, device=device, dtype=torch.long)
        pred_noise = model(x, t)

        alpha_t = schedule['alphas'][step]
        alpha_bar_t = schedule['alpha_bars'][step]
        beta_t = schedule['betas'][step]

        if step > 0:
            noise = torch.randn_like(x)
        else:
            noise = torch.zeros_like(x)

        x = (
            (1.0 / torch.sqrt(alpha_t))
            * (x - ((1.0 - alpha_t) / torch.sqrt(1.0 - alpha_bar_t)) * pred_noise)
            + torch.sqrt(beta_t) * noise
        )

    return x

### Autoencoder Training Helper
This helper trains only the autoencoder reconstruction stage.

Objective:
- Minimize reconstruction error between input and decoded output.

Pseudocode:
1. Forward: `x_hat, z = AE(x)`
2. Compute `MSE(x_hat, x)`
3. Backpropagate and update AE parameters
4. Log epoch-average reconstruction loss

In [ ]:
def train_autoencoder(model, loader, epochs=20, lr=1e-3):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    history = []
    model.train()

    for epoch in range(epochs):
        running = 0.0
        n_batches = 0
        for x, _ in tqdm(loader, desc=f'AE epoch {epoch + 1}/{epochs}', leave=False):
            x = x.to(device)
            x_hat, _ = model(x)
            loss = 0.5 * F.mse_loss(x_hat, x) + 0.5 * F.l1_loss(x_hat, x)

            opt.zero_grad()
            loss.backward()
            opt.step()

            running += loss.item()
            n_batches += 1

        avg = running / max(n_batches, 1)
        history.append(avg)
        print(f'AE epoch {epoch + 1:02d}/{epochs} | recon loss: {avg:.4f}')

    return history

### Diffusion Training Algorithm (for `train_diffusion`)
This block trains the denoiser to predict the exact noise used in `q_sample`.

Loss:
$$\mathcal{L}_{diff} = \mathbb{E}_{x_0, t, \varepsilon}\left[\lVert \varepsilon - \varepsilon_\theta(x_t, t) \rVert^2\right]$$

Pseudocode:
1. Sample clean target (`x0`) in pixel space or latent space.
2. Sample random timestep `t` and noise `eps`.
3. Build `x_t` with forward noising.
4. Predict noise `eps_pred`.
5. Minimize `MSE(eps_pred, eps)` and update model.

In [ ]:
def train_diffusion(model, loader, schedule, epochs=20, lr=2e-4, encode_fn=None):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    history = []
    model.train()

    for epoch in range(epochs):
        running = 0.0
        n_batches = 0

        for x, _ in tqdm(loader, desc=f'Diff epoch {epoch + 1}/{epochs}', leave=False):
            x = x.to(device)
            if encode_fn is not None:
                with torch.no_grad():
                    x = encode_fn(x)

            # TODO START - Diffusion training step
            # 1) Determine batch size and sample random diffusion steps.
            batch_size = x.shape[0]
            sample_diffusion = torch.randint(0, schedule.num_timesteps, (batch_size,), device=x.device).long()
            # 2) Use q_sample to obtain x_t and the target noise.
            q_sample = torch.rand_like(x)
            x_t = schedule.q_sample(x, sample_diffusion, noise=q_sample)            
            # 3) Predict the noise with model(x_t, t).
            pred_noise = model(x_t, sample_diffusion)
            # 4) Compute the MSE loss against the sampled noise.
            MSE_loss = F.mse_loss(pred_noise, q_sample)
            # 5) Zero gradients, backpropagate, and step the optimizer.
            opt.zero_grad()
            MSE_loss.backward()
            opt.step()
            # 6) Update the running loss and batch counter.
            loss = MSE_loss.item()
            running += loss
            n_batches += 1

        avg = running / max(n_batches, 1)
        history.append(avg)
        print(f'Diff epoch {epoch + 1:02d}/{epochs} | loss: {avg:.4f}')

    return history


## 1 - Pixel-space diffusion on MNIST
Train a denoiser directly on image space (1 x 32 x 32).

This setup uses a shorter diffusion horizon and a wider denoiser so recognizable digits appear in fewer epochs.

In [ ]:
mnist_train_loader, mnist_test_loader = build_mnist_loaders(
    batch_size=128,
    train_limit=5000,
    test_limit=5000,
)

x_real, _ = next(iter(mnist_train_loader))
show_image_grid(x_real, channels=1, title='MNIST real samples', n_show=25)

Configure and train the MNIST diffusion model.

In [ ]:
# TODO START
# 1) Instantiate the pixel-space diffusion process.
# 2) Instantiate the denoising network and optimizer.
# 3) Train the model on MNIST by sampling timesteps and predicting noise.
# 4) Track the average loss per epoch in pixel_history.
pixel_diffusion = GaussianDiffusion(device=device)
pixel_model = PixelUNet().to(device=device)
optimizer = torch.optim.Adam(pixel_model.parameters(), lr=2e-4)

epochs = 20
pixel_history = train_diffusion(pixel_model,mnist_train_loader,pixel_diffusion)

plot_curve(pixel_history, title='Pixel-space Diffusion Loss')


Generate MNIST samples from the trained diffusion model.

In [ ]:
# TODO START
# Sample 16 images with the trained pixel-space diffusion model.
sampled = pixel_diffusion.p_sample_loop(pixel_model, shape=(16, 1, 28, 28))
# TODO END
show_image_grid(sampled, title='Generated Pixel Samples', n_show=16)


## 2 - Latent diffusion on MNIST
Two-stage pipeline:
1. Train an autoencoder to map images <-> latent tensors.
2. Train diffusion model on latent tensors instead of pixels.

The autoencoder is wider than the first draft so decoded latent samples recover digit structure more easily.

Train and inspect autoencoder reconstructions.

In [ ]:
# TODO START
print("Training VAE...")

# 1) Define a VAE loss with reconstruction and KL terms.
# 2) Instantiate the VAE and its optimizer.
# 3) Train the VAE on MNIST and store the epoch losses in ae_history.
def vae_loss_function(recon_x, x, mu, logvar, kl_weight=0.01):
    reconstruction_loss = F.mse_loss(recon_x, x)
    kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    kl_loss = kl_weight * kl_loss / x.size(0)  # Normalize by batch size
    return reconstruction_loss + kl_loss


vae = VAE().to(device=device)
optimizer_ae = torch.optim.Adam(vae.parameters(), lr=1e-3)
ae_history = train_autoencoder(vae, mnist_train_loader, epochs=20, lr=1e-3)
# TODO END


plot_curve(ae_history, title='VAE Loss')


Train diffusion model in latent space and decode generated latents back to image space.

In [ ]:
# freeze autoencoder while training latent diffusion
for p in vae.parameters():
    p.requires_grad = False
vae.eval()

# TODO START
# 1) Instantiate the latent diffusion process, denoiser, and optimizer.
# 2) Encode MNIST images into latents with the frozen VAE.
# 3) Train the latent denoiser to predict the sampled noise.
# 4) Track the average loss per epoch in latent_history.
latent_diffusion = GaussianDiffusion(device=device)
latent_model = LatentDenoiseNetwork().to(device=device)
optimizer_lat = torch.optim.Adam(latent_model.parameters(), lr=2e-4)

epochs_lat = 20
latent_history = train_diffusion(
    model=latent_model,
    loader=mnist_train_loader,
    schedule=latent_diffusion,
    encode_fn=lambda x: vae.encoder(x)[0]  # Use mu as latent representation
)

# TODO END

if latent_diffusion is None or latent_model is None or optimizer_lat is None or latent_history is None:
    raise NotImplementedError('Set up and train the latent diffusion model.')

plot_curve(latent_history, title='Latent Diffusion Loss')


Sample latent tensors with diffusion and decode to MNIST images.

In [ ]:
# TODO START
# Sample latent codes with the diffusion model and decode them with the VAE decoder.
z_sampled = latent_diffusion.p_sample_loop(latent_model, shape=(25, 4, 4, 4))
recon = vae.decoder(z_sampled)
# TODO END


show_image_grid(recon, title='Generated Latent Samples', n_show=25)


## Final reflection
1. How did training stability differ between pixel-space diffusion and latent diffusion?
2. What did the latent model gain or lose relative to direct pixel diffusion?
3. Which step in the training loop was most critical to get correct?
4. If you could tune one variable first, what would it be and why?

Answers:
1. Latent diffusion was usually faster per step after autoencoder training.
2. Latent space improved efficiency but could lose fine detail.
3. Correct noising target alignment (`pred_noise` vs true `noise`) was critical.
4. Number of diffusion steps (`T`) is a high-impact first knob.